In [0]:
## raw data folder = /Volumes/tobias/staging/bufferdata/raw_data/master_data/
## mes_data, plc_data
## table folder = tobias.default.error_modular, tobias.default.timebased_delta
## tobias.default.lengthbased_delta, tobias.default.master_udm_delta, tobias.default.mes_udm_delta_v2
## buffer directory = /Volumes/tobias/staging/bufferdata/plc_CRM_modular_mapping/buffer_plc/

In [0]:
### Code to create temp views from parquet files

out_dir1 = "/Volumes/tobias/staging/bufferdata/raw_data/plc_data/"
df= spark.read.parquet(out_dir1)
df.createOrReplaceTempView("timebased_delta")

out_dir2 = "/Volumes/tobias/staging/bufferdata/raw_data/mes_data/"
df= spark.read.parquet(out_dir2)
df.createOrReplaceTempView("mes_udm_delta_v2")

out_dir3 = "/Volumes/tobias/staging/bufferdata/raw_data/master_data/"
df= spark.read.parquet(out_dir3)
df.createOrReplaceTempView("master_udm_delta")

# COMMAND ----------

In [0]:
# Databricks notebook source
import os
from time import sleep
from pyspark.sql.types import *
from pyspark.sql.functions import input_file_name, current_timestamp, pandas_udf, col, first, when
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pyspark.sql.functions import col, expr, broadcast
import datetime as dt 
import datetime
import traceback
import pytz

#current_catalog = spark.conf.get("spark.databricks.sql.initial.catalog.name")
current_catalog = "tobias"


class MappingException(Exception):
  pass

# COMMAND ----------

def insert_error_modular(product_id, flag, start_time_machine, exit= 1):
  nb_errors = spark.sql(f''' select count(*) as error_count from tobias.default.error_modular where ProductId = '{product_id}' and Domain = '{product_id[-11:-1]}' ''').collect()[0]['error_count']
  if nb_errors == 0:
    productErr = [(product_id,product_id[-11:-1], dt.datetime.now(), flag,start_time_machine,1)]
    df = spark.createDataFrame(productErr,['ProductId', 'Domain','DateProcess','Flag','startdatetimemachinespecific','error_counter'])
    df.write.insertInto('tobias.default.error_modular',overwrite = False)
  else : 
    spark.sql(f''' update tobias.default.error_modular set flag = '{flag}', error_counter = error_counter+1,DateProcess = current_date() where ProductId = '{product_id}' and Domain = '{product_id[-11:-1]}' ''')

  if exit == 1:
    raise MappingException(f'Error from data ({flag}) -> insert into error.modular') # Exception(f'Error from data ({flag}) -> insert into error.modular')


# COMMAND ----------

### Code to create temp views from parquet files

# out_dir1 = "/Volumes/tobias_dev/staging/bufferdata/test/plc_data/"
# df= spark.read.parquet(out_dir1)
# df.createOrReplaceTempView("timebased_delta")

# out_dir2 = "/Volumes/tobias_dev/staging/bufferdata/test/mes_udm/"
# df= spark.read.parquet(out_dir2)
# df.createOrReplaceTempView("mes_udm_delta_v2")

# out_dir3 = "/Volumes/tobias_dev/staging/bufferdata/test/master_udm/"
# df= spark.read.parquet(out_dir3)
# df.createOrReplaceTempView("master_udm_delta")

# COMMAND ----------

def map_coil_positions(ProductId:str, Startdatetime:str,Enddatetime:str, target_write_dir:str, product_mes_ref_df:pd.DataFrame, buffer_dir:str, sensor_relative_position_df:pd.DataFrame):
  startTime = dt.datetime.now()

                               #### optimization 1 ####
    # product_mes_ref_df is a pandas df, it first converted to spark df then again back 
    # to pandas df. This is not necessary. We can directly use the pandas df to for insert_error_modular function 

  #MES data to use as reference data for plc
  #spark.createDataFrame(product_mes_ref_df).createOrReplaceTempView("temp_data")
  #check if there is an anomalie with the duration (less than 60 seconds to process one coil)
  #pdt = spark.table("temp_data").toPandas()
  pdt = product_mes_ref_df.reset_index(drop=True) 
  StartDateTimeMachineSpecific = int(pdt['StartDateTimeMachineSpecific'].max().replace(tzinfo=dt.timezone.utc).timestamp())

  if(pdt['unix_EndDateTimeMachineSpecific'][0]-pdt['unix_StartDateTimeMachineSpecific'][0] < 60):
    insert_error_modular(ProductId, 'Duration time < 60 sec', StartDateTimeMachineSpecific)
  
  #MES data to use as reference data for plc
  spark.createDataFrame(product_mes_ref_df).createOrReplaceTempView("temp_data")
  
  #read plc data data from the buffer dir
  spark.read.format("parquet").load(buffer_dir).where(col("ProductId")==ProductId).drop(col("ProductId")).createOrReplaceTempView("plc_retrieve_coil")

             
  #join plc and mes data
  #spark.sql(f"""-- get sensors data for the product
  #create or replace temporary view plc_mes_udm_retrieve_unit as
  #SELECT DISTINCT
  #    -- PLC Columns
  #    TagId, Value, Timestamp, unix_timestamp, Date, Domain, Flag_ValueInRange,
  #    -- MES Columns
  #    ProductId, Machine, StartDateTimeMachineSpecific, EndDateTimeMachineSpecific, unix_StartDateTimeMachineSpecific, unix_EndDateTimeMachineSpecific,
  #    Length, Width, Thickness, Weight
  #    FROM temp_data src, plc_retrieve_coil plc WHERE TagId IS NOT NULL
  #    """) 
  
            ### optimization 2nd ###
        #  if we broadcast mes data, then whole shuffle will not happen. Since PLC data has no duplicates then there  is no need of DINSTINCT. Also, prefixing columns for ambugity and readability.


  spark.sql(f"""-- get sensors data for the product
  create or replace temporary view plc_mes_udm_retrieve_unit as
  SELECT /*+ BROADCAST(src) */
      -- PLC Columns
      plc.TagId, plc.Value, plc.Timestamp, plc.unix_timestamp, plc.Date, plc.Domain, plc.Flag_ValueInRange,
      -- MES Columns
      src.ProductId, src.Machine, src.StartDateTimeMachineSpecific, src.EndDateTimeMachineSpecific,
      src.unix_StartDateTimeMachineSpecific, src.unix_EndDateTimeMachineSpecific,
      src.Length, src.Width, src.Thickness, src.Weight
  FROM temp_data src
  CROSS JOIN plc_retrieve_coil plc
  WHERE plc.TagId IS NOT NULL
""")
  
  # parameters depending on CRM2 or CRM3

  if ProductId[-11:]=='GN-CR-CRM2_':
    #print('CRM2')
    Zone = 'GN-CR-CRM2'
    Sensors = ["GN_CR_CRM2_LengteGewalstH1","GN_CR_CRM2_LengteGewalstH2","GN_CR_CRM2_PasHuidig"]
    even_pass = "GN_CR_CRM2_LengteGewalstH1"
    uneven_pass = "GN_CR_CRM2_LengteGewalstH2"
    pass_number = "GN_CR_CRM2_PasHuidig"

  elif ProductId[-11:]=='GN-CR-CRM3_':
    # print('CRM3')
    Zone = 'GN-CR-CRM3'
    Sensors = ["GN_CR_CRM3_strip_length_exit","GN_CR_CRM3_actuele_pas"]
    even_pass = "GN_CR_CRM3_strip_length_exit"
    uneven_pass = "GN_CR_CRM3_strip_length_exit"
    pass_number = "GN_CR_CRM3_actuele_pas"

  # if product is not CRM2 or CRM3, it's an error
  else :
    insert_error_modular(ProductId, 'Not from CRM process', StartDateTimeMachineSpecific)
    

  # pivot to have only the data for the sensors that we need for product position

        #### optimization 3 - unnecessary order by Timestamp at the start

  df = spark.table("plc_retrieve_coil") 
  #.orderBy('Timestamp')

  # Sensor values to float
  #df = df.withColumn("Value",df["Value"].cast('float'))
  #df = df.withColumn("Timestamp",df["Timestamp"].cast('Timestamp'))

        #### optimization 4 - it will not run this two cast in the plan. Also, only required column are selected
             #( column puring) 

  df = df.select(col("Timestamp").cast("timestamp").alias("Timestamp"), 
                 col("TagId"), col("Value").cast("float").alias("Value"))
  
  # Replace pivot with conditional aggregation (eliminates the pivot bottleneck)
  # This uses first() with when() conditions instead of pivot, which is Photon-accelerated
  # Note: filter is applied AFTER aggregation to exclude timestamps that don't have all required sensors
  pivot = df.groupBy("Timestamp") \
    .agg(
        first(when(col("TagId") == even_pass, col("Value")), ignorenulls=True).alias("evenPass"),
        first(when(col("TagId") == uneven_pass, col("Value")), ignorenulls=True).alias("unevenPass"),
        first(when(col("TagId") == pass_number, col("Value")), ignorenulls=True).alias("passNumber")
    ) \
    .orderBy('Timestamp') \
    .toPandas()


  # if we don't have data, it's an error
  if pivot.empty:
    insert_error_modular(ProductId, 'No data for this product (pivot result empty)', StartDateTimeMachineSpecific)

  # delete wrong data in CRM2 (residues from the previous coil)
  # we filter data from the first zero in the first pass number equals to 1
  df_passOne = pivot[pivot['passNumber'] == 1]

  # Validation Checkpoint
  if df_passOne.empty:
    insert_error_modular(ProductId, 'No first Pass PLC Data for this product (df_passOne result empty)', StartDateTimeMachineSpecific)


  if Zone == 'GN-CR-CRM2' :
    min_unevenPass = df_passOne['unevenPass'].min()
    filtered_index = pivot.index[(pivot['unevenPass'] == min_unevenPass) & (pivot['passNumber'] == 1)]
    # Validation of Index
    if filtered_index.empty:
      insert_error_modular(ProductId, 'Filtering condition not matched for first Pass', StartDateTimeMachineSpecific)
    df_filtered = pivot[filtered_index[0]:]

  # no data to delete for CRM3
  if Zone =='GN-CR-CRM3':
    df_filtered = pivot

  # there is no pass number equals to 0
  df_filtered = df_filtered[df_filtered['passNumber'] != 0]

  #delete NaN values and interpolation

  df_filtered.set_index('Timestamp',inplace=True)
  df_filtered.evenPass.interpolate(method="time", inplace = True)
  if Zone == 'GN-CR-CRM2' :
    df_filtered.unevenPass.interpolate(method="time", inplace = True)
    df_filtered.unevenPass.fillna(method="bfill", inplace = True)
  df_filtered.evenPass.fillna(method="bfill", inplace = True)
  df_filtered.passNumber.interpolate(method="time", inplace = True)
  df_filtered.passNumber.fillna(method="bfill", inplace = True)

  # convert float to int
  df_filtered['passNumber'] = df_filtered['passNumber'].astype(int)

  # calcul of the product position is explain in the document provide for mapping logic :

  grouped = df_filtered.groupby(by="passNumber")

  groups = []
  for name, group in grouped:
    modGroup = group
    if Zone == 'GN-CR-CRM2' :
      x = modGroup["unevenPass"]
    else :
      x = modGroup["evenPass"]
    modGroup["ProductPosition"] = modGroup["evenPass"].max() - modGroup["evenPass"] if (modGroup["passNumber"] % 2)[0].item() == 0 else x
    groups.append(modGroup)

  positiondf = pd.concat(groups)
  df = positiondf.reset_index()

  # for some coils (in CRM3) we need to delete some wrong values (like peak of extreme values)
  # so we compare the number of data between 2 products positions equals to 0 (add in a list "lst") thanks to the index
  ### It's only applicable for CRM3 
  if Zone == 'GN-CR-CRM3' :
    
    result = df.index[df["ProductPosition"] == 0]
    lst = []
    for i in range(0,len(result)):
      lst.append([result[i]-result[i-1],result[i],result[i-1]])
    lst.reverse()
    
    # when there is less than 1000 values (and more than 1), it's not coherent values so we delete it
    for j in lst:
      if j[0] < 1000 and j[0] > 1 :
        # print("j: ",j," / j[1]: ",j[1]," / j[2]: ",j[2],"from : ",df["Timestamp"][j[2]]," to : ",df["Timestamp"][j[1]])
        df = df.drop(df.index[j[2]:j[1]])

  sparkDF = spark.createDataFrame(df)

                ####  optimization 5 - 
                #No need for where clause cause plc_mes_udm_retrieve_unit only has required productId. Also, no need to sort Timestamp because in next step we are doing a join which shuffles the data anyway and no need to cast Timestamp to timestamp type. Also, no need to sort Timestamp  
                ###

  plc_mes_df = spark.sql('''
    SELECT TagId, Timestamp, Value, Date, unix_timestamp, Domain, ProductId, unix_StartDateTimeMachineSpecific, unix_EndDateTimeMachineSpecific, Flag_ValueInRange, Length
    FROM plc_mes_udm_retrieve_unit
    --WHERE productid in (select productId from temp_data);
  ''')
  #.withColumn("Timestamp", col("Timestamp").cast("timestamp")).sort("Timestamp")
  # print(plc_mes_df.count())

             #### optimization 6 - unnecessary cast and sorting Timestamp. Can broadcast plc_mes_df because size around 5MB

  first_merge_df = plc_mes_df \
    .join(sparkDF, ["Timestamp"], "right_outer") \
    #.withColumn("Timestamp",col("Timestamp").cast("timestamp")) \
    #.sort("Timestamp")


  data_master = spark.createDataFrame(sensor_relative_position_df)
 
  # rename column identifier into tagid for the join with dataframe
  data_master = data_master.withColumnRenamed("Identifier","TagId")

             #### optimization 7 - unnecessary sorting of Timestamp. Can broadcast data_master because of very small size

  second_merge_df = broadcast(data_master)\
    .join(first_merge_df, ["TagId"], "right_outer")\
    #.sort("Timestamp")

  #second_merge_df.toPandas().plot(x='Timestamp', y='ProductPosition',figsize=(35, 8))

  final_df = second_merge_df.withColumn("delta",second_merge_df["ProductPosition"] - second_merge_df["Sensor_Position_Relative"])

  # rename column identifier into tagid for the join with dataframe
  final_df = final_df.withColumnRenamed("ProductPosition","Position").withColumnRenamed("delta","ProductPosition")

  final_df = final_df[final_df['ProductPosition'] >= 0]

  # put in temp view
  final_df.createOrReplaceTempView("result")

  # Use the cached DataFrame directly instead of cacheTable (not supported on Serverless)
  datum = spark.sql(''' select max(length) as mL,max(ProductPosition) as mPP from result ''').first()

  ### Compare the calculated and actual coil lengths — if the deviation exceeds the threshold, insert into error_modular; otherwise, proceed with saving the processed data.
  if (datum.mL != None):
    length = datum.mL
    maxPP = datum.mPP
    ecart = (float(length) - float(maxPP)) / float(length) * 100

    if (abs(ecart) > 5):
      insert_error_modular(ProductId, 'Ecart length > 5%', StartDateTimeMachineSpecific)
  
              #### optimization - repartition(1) this will only create 1 file per productId 

  #write result to a dir
  spark.table("result").repartition(1).write.format("parquet").mode("append").save(target_write_dir+f'/productId={ProductId}')

  spark.sql(f''' DELETE FROM tobias.default.error_modular where productid = '{ProductId}' ''')
  endTime = dt.datetime.now()
  return {'time':(endTime - startTime).total_seconds(),'maxPP':maxPP, 'length':length, 'deviation':ecart}

## func end
# COMMAND ----------

# DBTITLE 1,Coils to Process
### THIS CELL CREATE A A DATAFRAME HAVING COILS TO BE PROCESSED

spark.sql(f'''create or replace temporary view COILS_TO_RETRIEVE_CRM
as
  select src.* 
  from mes_udm_delta_v2 src
  where 
    Asset_Tree IN ('GN-CR-CRM2')
    and to_date(startdatetimemachinespecific)= "2025-07-08"
    ORDER BY unix_StartDateTimeMachineSpecific ASC
    ''')

df_CRM = spark.table("COILS_TO_RETRIEVE_CRM")

for convert_col in ['UnderWindEntry', 'UnderWindExit', 'HasBeginEndChanged', 'NumberOfPasses', 'NumberOfPassesProcessed', 'Length', 'Width', 'Thickness', 'WidthPositionSegmInBegin', 'WidthPositionSegmInEnd', 'SpeedSetpoint', 'SpeedRealised', 'MetalUnitMaster_FK', 'MetalUnit_FK', 'MetalMovemenet_FK']:
  df_CRM = df_CRM.withColumn(convert_col,col(convert_col).cast(DoubleType()))


df_CRM_pd = df_CRM.toPandas()
if len(df_CRM_pd)>0:
  s = df_CRM_pd['StartDateTimeMachineSpecific'].min().strftime("%Y-%m-%d %H:%M:%S")
  e = df_CRM_pd['EndDateTimeMachineSpecific'].max().strftime("%Y-%m-%d %H:%M:%S")
  domains = ",".join([f"'{domain}'" for domain in df_CRM_pd['Asset_Tree'].unique()])
else:
  dbutils.notebook.exit("No Products to process. Quitting...")

#df_CRM_pd.display()
products = df_CRM_pd['ProductId'].unique()
print(f"Processing {len(products)} products between {s} and {e} in {domains} ")

# COMMAND ----------

# DBTITLE 1,Saving PLC Data to buffer Directory
### Save PLC data to the buffer directory (only for the fetched coils in above cdm line) to enable faster retrieval, as querying the large timebased_delta table for each product is time-consuming.

buffer_dir = f"/Volumes/{current_catalog}/staging/bufferdata/plc_CRM_modular_mapping/buffer_plc/"
dbutils.fs.rm(buffer_dir,True)
buffer_sql = f"""
SELECT 
    src.TagId, src.Value, src.Timestamp, src.unix_timestamp, src.Date, 
    src.Domain, src.Flag_ValueInRange, mes.ProductId
FROM 
    timebased_delta src
JOIN 
    COILS_TO_RETRIEVE_CRM mes 
    ON mes.StartDateTimeMachineSpecific <= src.Timestamp 
    AND src.Timestamp <= mes.EndDateTimeMachineSpecific
    AND mes.Asset_Tree = src.Domain
WHERE 
    src.Timestamp BETWEEN '{s}' AND '{e}'
    AND src.Domain IN ({domains})
    AND (src.Domain != 'GN-CR-CRM2' 
        OR src.TagId IN (SELECT identifier FROM master_udm_delta WHERE DOMAIN = 'GN-CR-CRM2' AND (Asset_Tree != 'GN-CR-CRM2-83' OR identifier = 'GN_CR_CRM2_LengteGewalstH2'))
    )
""" 

#If there is  no data, quit the job
if spark.sql(buffer_sql).count()  ==  0:
    #### Insert all the product into error modular table
    for product, starttime in zip(df_CRM_pd['ProductId'], df_CRM_pd['StartDateTimeMachineSpecific']):
        starttime = int(starttime.replace(tzinfo=dt.timezone.utc).timestamp())
        # starttime = to_spark_datetime(starttime)
        insert_error_modular(product, 'No PLC Data', starttime,0)
        print(f"{product} Inserted into Error Modular")

    dbutils.notebook.exit("No Timebased data. Quitting...")
else:
    spark.sql(buffer_sql).write.format("parquet").mode("overwrite").partitionBy("ProductId").save(buffer_dir)



sensor_data_df = spark.sql(f"""
SELECT Identifier, Sensor_Position_Relative
FROM master_udm_delta where Domain in ({domains})
""").toPandas()

# COMMAND ----------

### WE ARE PROCESSING EACH COIL FROM THE ABOVE LIST OF COILS WE GOT FROM df_CRM_pd IN A FOR LOOP AND SAVING THE PROCESSED DATA IN WRITEDIR DIRECTORY. 

WriteDir = f"/Volumes/{current_catalog}/staging/bufferdata/plc_CRM_modular_mapping/processed"

dbutils.fs.rm(WriteDir, True)
dbutils.fs.mkdirs(WriteDir)

result = {}
_processed = 0
_errors = 0
N = len(df_CRM_pd)
for product, start, end in zip(df_CRM_pd['ProductId'].values, df_CRM_pd['StartDateTimeMachineSpecific'].values,df_CRM_pd['EndDateTimeMachineSpecific'].values):
  startTime = datetime.datetime.now()
  product_mes_ref_df = df_CRM_pd.loc[df_CRM_pd.ProductId==product]
  try:
      result[product] = map_coil_positions(product,start,end,WriteDir,product_mes_ref_df, buffer_dir,sensor_data_df)
      print("Total time taken to process the product: ",datetime.datetime.now() - startTime)
      print(f"Product {product} processed : {result[product]}")
  except MappingException as error_message:
      print(f"Product {product} failed because of: ", error_message)
      _errors += 1
  except Exception as e:
      print(f"Product {product} failed because of: {e}")
      _errors += 1
  _processed += 1
  print(f"Processed {_processed} / {N} **** Errors: {_errors}")

# COMMAND ----------

# DBTITLE 1,Inserting Processed Data into Final Destination Table
### Now inserting all the successfully processed data into the destination table- lengthbased_delta table

"""Check if the directory exists and contains parquet files"""
files = dbutils.fs.ls(WriteDir)

if len(files) == 0:
  dbutils.notebook.exit("Every products failed. Quitting...")

schema = StructType([
    StructField("TagId", StringType(), True),
    StructField("Timestamp", TimestampType(), True),
    StructField("Value", DoubleType(), True),
    StructField("Date", DateType(), True),
    StructField("unix_timestamp", LongType(), True),
    StructField("Domain", StringType(), True),
    StructField("ProductId", StringType(), True),
    StructField("unix_StartDateTimeMachineSpecific", LongType(), True),
    StructField("unix_EndDateTimeMachineSpecific", LongType(), True),
    StructField("Flag_ValueInRange", BooleanType(), True),
    StructField("PassNumber", LongType(), True),  # Change to LongType to match INT64
    StructField("ProductPosition", DoubleType(), True),
    StructField("Length", DoubleType(), True)
])

df = spark.read.schema(schema).parquet(WriteDir)
df = df.withColumn("PassNumber", df["PassNumber"].cast(IntegerType()))  # Cast to IntegerType
df.createOrReplaceTempView("result")

result_df = spark.sql("""
    INSERT INTO tobias.default.lengthbased_delta (TagId,Timestamp,Value,Date,unix_timestamp,Domain,ProductId,unix_StartDateTime,unix_EndDateTime,Flag_ValueInRange,PassNumber,ProductPosition,Length)
      SELECT /*+ REPARTITION(1)*/
        source.TagId,
        source.Timestamp,
        source.Value,
        source.Date,
        source.unix_timestamp,
        source.Domain,
        source.ProductId,
        source.unix_StartDateTimeMachineSpecific,
        source.unix_EndDateTimeMachineSpecific,
        source.Flag_ValueInRange,
        source.PassNumber,
        round(source.ProductPosition,2) as ProductPosition,
        source.Length FROM result source
          """)

display(result_df)

In [0]:
%sql
select * from tobias.default.lengthbased_delta limit 10